# Part B: does dropout produce self-repair? The training sweep (Colab)

Eighteen small GPT-2 models (6 layers, 8 heads, d_model 384) trained from scratch on TinyStories: three regimes (`none`, `standard` dropout 0.1, `drophead` 0.1) × six seeds. Each run measures validation loss and the pooled self-repair fraction at nine points during training. The design is fixed in `docs/post/README.md` §4.4–4.6, committed before any run.

**Before running:** Runtime → Change runtime type → **A100 GPU**. Add the `WANDB_API_KEY` secret. Each run takes about 20–25 minutes, so the whole grid is about 7 hours of A100 time. It does not have to run in one session: a grid sweep hands out each (arm, seed) cell once, and cell 10 picks up the cells not yet run.

The order matters the first time: data (cell 6), smoke run (cell 7), timing (cell 8), register the sweep once (cell 9), then the agent (cell 10). In later sessions run cells 1–6, paste the sweep id into cell 10, and run it.

## 1. Check the GPU
Expect an A100. If this prints nothing, the runtime type is still CPU.

In [ ]:
!nvidia-smi -L

## 2. Get the code
Clones the repo. While the repo is private, add a Colab secret `GITHUB_TOKEN` (a fine-grained GitHub token with read access to `kvenanzi/trophic-cascade`); once it is public no token is needed. `BRANCH` lets this run from a feature branch before it is merged; re-running the cell pulls the latest commit.

In [ ]:
import os, subprocess

from google.colab import userdata

REPO = "kvenanzi/trophic-cascade"
BRANCH = "study"
try:
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = None
URL = f"https://{TOKEN}@github.com/{REPO}.git" if TOKEN else f"https://github.com/{REPO}.git"
if not os.path.isdir("trophic-cascade"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, URL], check=True)
else:
    subprocess.run(["git", "-C", "trophic-cascade", "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", "trophic-cascade", "pull", "-q"], check=True)
!git -C trophic-cascade log --oneline -1

## 3. Make the package importable and install what Colab lacks
The clone goes on `sys.path` (for this notebook) and on `PYTHONPATH` (for the scripts run with `!python`), so `import trophic` reads the code straight from the clone. The pip line adds only what Colab does not ship. TransformerLens is capped below 4.0, which removed `HookedTransformer`.

In [ ]:
import sys
ROOT = os.path.abspath("trophic-cascade")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.environ["PYTHONPATH"] = ROOT
%pip install -q "transformer-lens>=2.16,<4.0" wandb wandb-workspaces
import torch, transformers, transformer_lens, trophic
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| transformers", transformers.__version__,
      "| transformer_lens", getattr(transformer_lens, "__version__", "?"), "| trophic from", os.path.dirname(trophic.__file__))

## 4. Log in to Weights & Biases
The API key comes from Colab Secrets (key icon on the left, `WANDB_API_KEY`, *Notebook access* on). Every run below logs to `within-noise/trophic-cascade`.

In [ ]:
import wandb
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()
%cd {ROOT}

## 5. Optional: keep the tokenized data on Google Drive
Tokenizing TinyStories takes several minutes. With Drive mounted, the token files are written there once and reused by later sessions.

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/trophic-cascade/tinystories"
else:
    DATA = "/content/data/tinystories"

## 6. Tokenize TinyStories
Writes `train.bin` and `validation.bin` (GPT-2 tokens, uint16). Skipped if they exist.

In [ ]:
from trophic.data import prepare
print(prepare(DATA))

## 7. Smoke run
A tiny budget on the real data, about a minute, logged as `drophead-s0-smoke`. It checks the whole path: training with head dropout, the conversion to TransformerLens, the self-repair measurement, and the W&B logging.

In [ ]:
!python scripts/05_train.py --smoke --arm drophead --data-dir {DATA}

## 8. Timing: fix the token budget (§4.4)
Times 200 training steps of each arm and applies the registered rule: the largest multiple of ten million tokens that the slowest arm trains in 20 minutes or less, capped at one pass over the training split. Logged as the run `timing`. Run once.

In [ ]:
!python scripts/05_train.py --timing --data-dir {DATA}
import json
BUDGET = json.load(open("outputs/timing.json"))["budget_tokens"]
print("token budget:", BUDGET)

## 9. Register the sweep (once)
Registers `sweeps/dropout.yaml` with the budget from cell 8 and prints the sweep id. **Copy the id into cell 10.** Do not re-run this cell in later sessions, since it would register a second sweep.

In [ ]:
!python scripts/06_sweep.py --create --tokens {BUDGET}

## 10. Run the agent
`count` is how many cells this session should run; 18 runs the whole grid. After a disconnect, re-run cells 1–6 and this one; the agent continues with the cells not yet started. A run that crashed is not re-issued automatically, and the summary script reports it as missing.

In [ ]:
import gc, torch
from trophic.train import TrainConfig, train

SWEEP_ID = "within-noise/trophic-cascade/PASTE_ID_HERE"

def run_one():
    train(TrainConfig(data_dir=DATA, output_dir="models", tags=["sweep", "colab"]), sweep=True)
    gc.collect(); torch.cuda.empty_cache()

wandb.agent(SWEEP_ID, function=run_one, count=18)

## 11. What to look at
- The **sweep page**: the parallel-coordinates chart of `arm` → `final/self_repair_pooled`, and the runs table grouped by `arm`.
- In the project workspace, group runs by `arm`: `train/loss`, `measure/val_loss`, and `measure/self_repair_pooled` against step show how self-repair develops during training in each regime.
- Each run's **Tables → measurements** lists the nine measurement points; the artifact `measurements-<run>` holds the per-head values; `model-<run>` the weights.
- Back on the local machine: `uv run scripts/07_partb_summarize.py --sweep <id>` writes the paired analysis for H6 and H6c (`outputs/partb/summary.md`).